In [56]:
import os
os.listdir("../data")

['training.csv']

In [57]:
import pandas as pd
df = pd.read_csv("../data/training.csv", encoding='latin-1')

df.head()


,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [58]:
df.columns = ['sentiment','id','date','query','user','text']

In [59]:
df.head()

,sentiment,id,date,query,user,text
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [60]:
df = df[['sentiment','text']]
df.head()

,sentiment,text
0,0,is upset that he can't update his Facebook by ...
1,0,@Kenichan I dived many times for the ball. Man...
2,0,my whole body feels itchy and like its on fire
3,0,"@nationwideclass no, it's not behaving at all...."
4,0,@Kwesidei not the whole crew


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599999 entries, 0 to 1599998
Data columns (total 2 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   sentiment  1599999 non-null  int64 
 1   text       1599999 non-null  object
dtypes: int64(1), object(1)
memory usage: 24.4+ MB


In [62]:
df['sentiment'].value_counts()

sentiment
4    800000
0    799999
Name: count, dtype: int64

In [63]:
df['sentiment'] = df['sentiment'].replace(4, 1)

In [64]:
df['sentiment'].value_counts()

sentiment
1    800000
0    799999
Name: count, dtype: int64

In [65]:
import re
import nltk
from nltk.corpus import stopwords

In [66]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [67]:
stop_words = set(stopwords.words('english'))

In [68]:
def clean_text(text):
    text = text.lower()  #Lowercase

    text = re.sub(r'https\s+', ' ', text)  # remove urls
    text = re.sub(r'@\w+ ', ' ', text)  #remove mentions
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) #remove special chars

    words = text.split()
    word = [word for word in words if word not in stop_words]

    return " ".join(words)

In [69]:
df['clean_text'] = df['text'].apply(clean_text)

In [70]:
df[['text', 'clean_text']].head()

,text,clean_text
0,is upset that he can't update his Facebook by ...,is upset that he can t update his facebook by ...
1,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...
2,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
3,"@nationwideclass no, it's not behaving at all....",no it s not behaving at all i m mad why am i h...
4,@Kwesidei not the whole crew,not the whole crew


In [71]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features = 5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_vec, y_train)

In [ ]:
y_pred = model.predict(X_test_vec)

In [ ]:
y_pred

array([0, 0, 0, ..., 1, 0, 1], shape=(320000,))

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.791553125


In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[124336  35158]
 [ 31545 128961]]


In [ ]:
def predict_sentiment(text):
    text_clean = clean_text(text)
    text_vec = vectorizer.transform([text_clean])
    result = model.predict(text_vec)[0]

    if result == 1:
        return "Positive"
    else:
        return "Negative"

In [ ]:
print(predict_sentiment("I Love this product"))
print(predict_sentiment("This is the worst thing ever"))

Positive
Negative


In [ ]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

In [ ]:
import os
os.listdir()

['model.pkl', 'sentiment_analysis.ipynb', 'vectorizer.pkl']